<a href="https://colab.research.google.com/github/krantik00/Training_Notebook1/blob/main/ECommerce_support_Chatbot_notebook_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot

In this MLS, we will build an ecommerce chatbot designed to streamline customer support. This AI-driven solution will efficiently handle customer inquiries related to:

Respond to prospective customers

1. Product Enquiry
2. Policy Enquiry

Information for existing customers

1. Delivery/Refund Tracking
2. Invoice Requests

Customer Feedback Collection

* For queries beyond its capabilities, the chatbot will automatically defer to
a human agent, ensuring a smooth and responsive experience.


# Setup

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from langchain_groq import ChatGroq

In [7]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool

/tmp/ipykernel_1176/3207111563.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings


The first step of our chatbot is to bring all the pre-existing modules we built for it. Namely, SQL tool,and RAG tool.

### SQL tool

Let's quickly build our SQL Agent

In [8]:
# Define the location of the SQLite database
db_loc = 'ecomm.db'

# Create a SQLDatabase instance from the SQLite database URI
db = SQLDatabase.from_uri(f"sqlite:///{db_loc}")

# Retrieve the schema information of the database tables
database_schema = db.get_table_info()

In [9]:
print(database_schema)


CREATE TABLE "Customers" (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone INTEGER NOT NULL, 
	address TEXT NOT NULL, 
	PRIMARY KEY (customer_id)
)

/*
3 rows from Customers table:
customer_id	first_name	last_name	email	phone	address
1	David	Taylor	david.taylor@yahoo.com	8010878513	1130 Birch Blvd, Chicago, AZ, 81011
2	Emily	Davis	emily.davis@outlook.com	6281972072	7897 Willow Ln, Chicago, CA, 55316
3	Michael	Smith	michael.smith@outlook.com	7579377501	2834 Oak St, Philadelphia, IL, 10112
*/


CREATE TABLE "Invoices" (
	order_id INTEGER NOT NULL, 
	invoice_date TEXT NOT NULL, 
	amount REAL NOT NULL, 
	invoice_url TEXT NOT NULL, 
	FOREIGN KEY(order_id) REFERENCES "Orders" (order_id)
)

/*
3 rows from Invoices table:
order_id	invoice_date	amount	invoice_url
2	2024-11-17 23:31:11	432.92	https://example.com/invoice/2
3	2024-11-25 16:20:51	473.85	https://example.com/invoice/3
4	2024-11-24 17:47:29	117.39	https://example.com/invoice

In [10]:
# Define the system message for the agent, including instructions and available tables
system_message = f"""You are a SQLite expert agent designed to interact with a SQLite database.
Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 100 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database..
You can order the results by a relevant column to return the most interesting examples in the database.
You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
You have access to tools for interacting with the database.
Only use the given tools. Only use the information returned by the tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
You are not allowed to make dummy data.

If the question does not seem related to the database, just return "I don't know" as the answer.
Before you execute the query, tell us why you are executing it and what you expect to find briefly.
Only use the following tables:
{database_schema}
"""

# Create a full prompt template for the agent using the system message and placeholders
full_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", '{input}'),
        MessagesPlaceholder("agent_scratchpad")
    ]
)

In [11]:
from google.colab import userdata
openai_api_key = userdata.get('GROQ_API_KEY')

In [12]:
llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=openai_api_key,
    temperature=0
)

In [13]:
# Create the SQL agent using the AzureChatOpenAI model, database, and prompt template
sqlite_agent = create_sql_agent(
    llm=llm,
    db=db,
    prompt=full_prompt,
    agent_type="openai-tools",
    agent_executor_kwargs={'handle_parsing_errors': True},
    max_iterations=5,
    verbose=True
)

#### Let's convert the sql agent into a tool that our top level agent can use.

In [14]:
@tool
def sql_tool(user_input):
    """
    Executes a SQL query using the sqlite_agent and returns the result.

    Args:
        user_input (str): a natural language query string explaining what information is required while also providing the necessary details to get the information.

    Returns:
        str: The result of the SQL query execution. If an error occurs, the exception is returned as a string.
    """
    try:
        # Invoke the sqlite_agent with the user input (SQL query)
        response = sqlite_agent.invoke(user_input)

        # Extract the output from the response
        prediction = response['output']

    except Exception as e:
        # If an exception occurs, capture the exception message
        prediction = e

    # Return the result or the exception message
    return prediction

#### Let's test the SQL tool that we have created

In [15]:
# # Test the sql_tool function with a sample user input
user_input = "Did Michael	Smith place an order?"
answer = sql_tool.invoke(user_input)
print(answer)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query_checker` with `{'query': "SELECT * FROM Orders WHERE customer_id IN (SELECT customer_id FROM Customers WHERE first_name = 'Michael' AND last_name = 'Smith') LIMIT 100"}`


SELECT * FROM Orders WHERE customer_id IN (SELECT customer_id FROM Customers WHERE first_name = 'Michael' AND last_name = 'Smith') LIMIT 100
Invoking: `sql_db_query` with `{'query': "SELECT * FROM Orders WHERE customer_id IN (SELECT customer_id FROM Customers WHERE first_name = 'Michael' AND last_name = 'Smith') LIMIT 100"}`


[(42, 3, 3, '2024-11-06 10:31:53', 'Delivered', 387.03, '3.91'), (62, 3, 7, '2024-11-06 10:17:26', 'Shipped', 234.5, '3.9')]Yes, Michael Smith placed an order.

> Finished chain.
Yes, Michael Smith placed an order.


### RAG Tool

#### Vector DB from Gdrive

Download the provided policy_docs.zip file from olympus, extract it and upload it to your Gdrive. The final folder contents should look like this: (names of the folders might change)

![image.png](attachment:image.png)

#### Fetch the DB from GDrive

First let's get the embedding model.

In [16]:
# Initialise the embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_1176/2347412379.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [17]:
import zipfile
import os

zip_path = '/content/policy_docs.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"'{zip_path}' extracted to '{extract_path}' successfully.")

# Optional: List contents to verify
# print(os.listdir(extract_path))

'/content/policy_docs.zip' extracted to '/content/' successfully.


In [18]:
# Load the persisted DB
persisted_vectordb_location = '/content/policy_docs'
#Create a Colelction Name
collection_name = 'policy_docs'
# Load the persisted DB
vector_store = Chroma(
    collection_name=collection_name,
    persist_directory=persisted_vectordb_location,
    embedding_function=embedding_model

)

/tmp/ipykernel_1176/644037121.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


#### RAG

In [19]:
qna_system_message = """
You are an assistant to a support agent. Your task is to provide relevant information.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context.
The context contains references to specific portions of documents relevant to the user's query, along with source links.
The source for a context will begin with the token ###Source

When crafting your response:
1. Select only context relevant to answer the question.
2. User questions will begin with the token: ###Question.
3. If the context provided doesn't answer the question respond with - "I do not have sufficient information to answer that"
4. If user asks for product - list all the products that are relevant to his query. If you don't have that product try to cross sell with one of the products we have that is related to what they are interested in.
You should get information about similar products in the context.

Please adhere to the following guidelines:
- Your response should only be about the question asked and nothing else.
- Answer only using the context provided.
- Do not mention anything about the context in your final answer.
- If the answer is not found in the context, it is very very important for you to respond with "I don't know."
- Always quote the source when you use the context. Cite the relevant source at the end of your response under the section - Source:
- Do not make up sources. Use the links provided in the sources section of the context and nothing else. You are prohibited from providing other links/sources.

Here is an example of how to structure your response:

Answer:
[Answer]

Source:
[Source]
"""

In [20]:
qna_user_message_template = """
###Context
Here are some documents and their source that may be relevant to the question mentioned below.
{context}

###Question
{question}
"""

In [21]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

Let's use all the context retrieved to generate the final answer

In [25]:
import os
from google.colab import userdata
from groq import Groq

# Set your API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [29]:
@tool
def rag(user_input: str) -> str:

    """
      Generates a response based on the user's input by retrieving relevant documents and using them as context for a GPT-4o model.

      Args:
          user_input (str): The input question or query from the user.

      Returns:
          response (str): Return the generated response or an error message if an exception occurs.

    """

    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content + "\n ###Source: " + d.metadata['source'] + "\n\n " for d in relevant_document_chunks]

    context_for_query = ". ".join(context_list)
    # print("context: ", context_for_query)  # Use this to understand what context is provided BTS and to debug.
    prompt = [
        {'role':'system', 'content': qna_system_message},
        {'role': 'user', 'content': qna_user_message_template.format(
            context=context_for_query,
            question=user_input
            )
        }
    ]

    try:
        response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=prompt
        )

        prediction = response.choices[0].message.content
    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return prediction


In [30]:
rag.invoke("What is the ram on the mobile phone?")

'4GB\n\nSource:\n(product_Smartphone_description.txt)'

In [31]:
rag.invoke("list all the keyboards available for sale?")

'Here are the keyboards available for sale:\n- Keyboard (Product ID: 5)\n\nSource:\nproduct_Keyboard_description.txt\nproduct_Keyboard_policy.txt'

In [32]:
rag.invoke("What is the refund policy for electronics ?")

'After inspection and approval of the return, a full refund will be processed within 7 business days, minus applicable cancellation fees, if any.\n\nSource:\nReturn Policy in Tablet Cancellation Policy Document (product_Tablet_policy.txt) \nand Return Policy in Monitor Cancellation Policy Document (product_Monitor_policy.txt)'

Now, let's build few other tools that our chatbot is gonna need. You don't have to pay a lot of attention to this code.

### Tools

* Since we dont have a logging functionality in colab, let's store our feedback in dataframes. when we switch to hugging face spaces,

* we are going to store them as datasets. we will change this code when we migrate to huggingface spaces

In [33]:
feedback_log = pd.DataFrame(columns=["timestamp", "intent", "customer_id", "feedback", "rating"])
@tool
def register_feedback(intent: str, customer_id: int, feedback: str, rating: int) -> str:
    """
    Logs customer feedback into the feedback log.

    Args:
        intent (str): The category of the support query (e.g., "cancel_order", "get_refund").
        customer_id (int): The unique ID of the customer.
        feedback (str): The feedback provided by the customer.
        rating(int): The rating provided by the customer out of 5

    Returns:
        str: Success message.
    """
    global feedback_log
    feedback_entry = {
        "timestamp": datetime.now(),
        "intent": intent,
        "customer_id": customer_id,
        "feedback": feedback,
        "rating": rating
    }
    feedback_log = pd.concat([feedback_log, pd.DataFrame([feedback_entry])], ignore_index=True)
    print("register_feedback success")
    return "Feedback registered successfully!"

deferred_cases = pd.DataFrame(columns=["timestamp", "customer_id", "query", "reason","intent"])

@tool
def defer_to_human(customer_id: str, query: str, intent: str, reason: str) -> str:
    """
    Logs customer details and the reason for deferring to a human agent.

    Args:
        customer_id (int): The unique ID of the customer whose query is being deferred.
        query (str): The customer's query or issue that needs human intervention.
        intent (str): The category of the support query (e.g., "order_tracking", "product_description",...etc)
        reason (str): The reason why the query cannot be resolved by the chatbot.

    Returns:
        str: Success message indicating the deferral was logged.
    """
    global deferred_cases
    case_entry = {
        "timestamp": datetime.now(),
        "customer_id": customer_id,
        "query": query,
        "reason": reason,
        "intent": intent
    }
    deferred_cases = pd.concat([deferred_cases, pd.DataFrame([case_entry])], ignore_index=True)
    print("defer_to_human success")
    return "Case deferred to human agent and logged successfully!"


@tool
def days_since(delivered_date: str) -> str:
    """
    Calculates the number of days since the product was delivered.

    Args:
        delivered_date (str): The date when the product was delivered in the format 'YYYY-MM-DD'.
    """
    try:
        # Convert the delivered_date string to a datetime object
        delivered_date = datetime.strptime(delivered_date, '%Y-%m-%d')
        today = datetime.today()

        # Calculate the difference in days
        days_difference = str((today - delivered_date).days)

        return days_difference
    except ValueError as e:
        return f"Error: {e}"


In [34]:
# Example Usage
register_feedback.invoke(input={
    "intent": "cancel_order",
    "customer_id": 1,
    "feedback": "Great experience!",
    "rating": 4
})

# Display Feedback Log
print("\nFeedback Log:")
print(feedback_log)


register_feedback success

Feedback Log:
                   timestamp        intent customer_id           feedback  \
0 2026-06-28 08:31:36.655709  cancel_order           1  Great experience!   

  rating  
0      4  


/tmp/ipykernel_1176/1124609176.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  feedback_log = pd.concat([feedback_log, pd.DataFrame([feedback_entry])], ignore_index=True)


In [35]:
# Initialize a log as a DataFrame
log = pd.DataFrame(columns=["id", "history"])

# Function to log actions
def log(customer_id: str,chat_history: list) -> None:
    global log
    log = pd.concat([log, pd.DataFrame({
        "id": customer_id,
        "history": chat_history
    })], ignore_index=True)
    print(f"log success")

In [36]:
# # Example usage
# cancel_order.invoke(input = {"order_id":101, "customer_id":1})
# process_refund.invoke(input = {"order_id":102, "refund_amount":50.0, "reason":"Defective product"})
# change_address.invoke(input = {"order_id":103, "new_address":"123 New Street, New City, 12345"})

# # Display log
# print("\nAction Log:")
# print(log)

You have access to the following tools, which you can use to address customer queries efficiently:

### **Available Tools:**
1. **`RAG tool`**: Retrieves relevant information from a database and answers questions related to product descriptions, cancellation policies and general policies.
2. **`sql_tool`**: Executes SQL queries on the database to retrieve necessary details such as order status, refund policies, invoices, and shipping information.
3. **`register_feedback`**: Logs feedback from the customer, including the support category and feedback received.
4. **`defer_to_human`**: Defers queries outside the chatbot’s capabilities to a human agent.
5. **`days_since`**: Determines the number of days since delivery.
---

Let's create a system prompt that will give proper instructions to the chatbot on how it should behave and respond.


In [37]:
system_message = f"""
You are an intelligent e-commerce chatbot designed to assist users with post-order queries. Your job is to efficiently handle customer inquiries related to:

Respond to,
prospective customers about 1. Product Enquiry 2. Policy Enquiry
Information for existing customers about 1. Delivery/Refund Tracking 2. Invoice Requests and other queries regarding their orders.

Gather only necessary information [customer id, email or phone + order id etc.] from the user to help them with their query.

Do not provide sql inputs to the sql tool - you only need to ask in natural language what information you need.
  - If at any point you cannot determine the next steps - defer to human with valid customer details. you do not have clearance to go beyond the scope the following flow.
If customer asks about a product, you should act as a sales representative and help them understand the product as much as possible and provide all the necessary information for them. You should also provide them the link to the product which you can get from the source of the information.
If a customer asks a query about a policy, be grounded to the context provided to you. if at any point you don't the right thing to say, politely tell the customer that you are not the right person to answer this and defer it to a human.
Any time you defer it to a human, you should tell the customer why you did it in a polite manner.

MANDATORY STEP:
After helping the customer with their concern,
- Ask if the customer needs help with anything else. If they ask for anything from the above list help them.
- If not, ask for feedback and log it using `register_feedback`.
Once the customer confirms they do not need any further assistance:
1. Ask for their feedback and rating out of 5.
2. Use the `register_feedback` tool to log:

### Handling Out-of-Scope Queries:
If the user's query, at any point is not covered by the workflows above:
- Respond:
  > "This is beyond my skill. Let me connect you to a customer service agent" and get necessary details from the customer and use the defer_to_human tool.
- End the conversation.
---
### Important Notes for the Model:
- Always aim to minimize the number of questions asked by retrieving as much information as possible from the database using `sql_tool` - retrieve customer information using any details provided and use it throughout the conversation.
- If you have deferred to human already once, do not defer again on the same topic.
- Please check user details if they are valid before you take any action. you should not defer to human without a customer id - get it from customer or SQL tool using email, name or phone number.
- Before you ask for user details, check if you have those details already in the conversation

### Very important instruction:
You are not allowed to provide other users information to the current user. You can only provide them information which pertains to them.


"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

In [38]:
tools = [sql_tool, rag, defer_to_human, register_feedback, days_since]

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


Understand the following code is not necessary.

In [39]:
# Function to process user input and generate a chatbot response
def chat_with_agent():
    print("Chatbot is ready! Type 'exit' to end the conversation.")
    conversation_history = [{"role": "assistant", "content": "Hey, how can I help you?"}]

    # Display the initial greeting
    print("Chatbot: Hey, how can I help you?")

    while True:
        # Get user input
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() == "exit":
            print("Chatbot: Thank you for chatting. Goodbye!")
            break

        # Add user input to conversation history
        conversation_history.append({"role": "user", "content": user_input})

        conversation_input = "\n".join(
            [f"{turn['role'].capitalize()}: {turn['content']}" for turn in conversation_history]
        )
        # Pass the history to the agent
        response = agent_executor.invoke({"input": conversation_input})

        # Add the chatbot's response to the history
        chatbot_response = response['output']
        conversation_history.append({"role": "assistant", "content": chatbot_response})

        # Display the chatbot's response
        print(f"Chatbot: {chatbot_response}")

Type Exit to exit the chat

In [ ]:
# Start the chati
chat_with_agent()

Chatbot is ready! Type 'exit' to end the conversation.
Chatbot: Hey, how can I help you?
You: what are the orders


> Entering new AgentExecutor chain...

Invoking: `sql_tool` with `{'user_input': 'What are the orders placed by customers?'}`




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query_checker` with `{'query': 'SELECT "order_id", "customer_id", "product_id", "order_date", "status", "price", "cancel_fee" FROM "Orders" LIMIT 100'}`


SELECT "order_id", "customer_id", "product_id", "order_date", "status", "price", "cancel_fee" FROM "Orders" LIMIT 100
Invoking: `sql_db_query` with `{'query': 'SELECT "order_id", "customer_id", "product_id", "order_date", "status", "price", "cancel_fee" FROM "Orders" LIMIT 100'}`


[(1, 23, 3, '2024-11-23 01:05:38', 'Cancelled', 387.03, '3.91'), (2, 8, 10, '2024-11-16 23:31:11', 'Delivered', 432.92, '3.45'), (3, 30, 6, '2024-11-24 16:20:51', 'Shipped', 473.85, '2.88'), (4, 36, 2, '2024-11-23 17:47:29', 'Delivered', 117.39, '0.19'),

/tmp/ipykernel_1176/1124609176.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  deferred_cases = pd.concat([deferred_cases, pd.DataFrame([case_entry])], ignore_index=True)


Would you like to ask anything else? If not, could you please provide your feedback and rating out of 5?

> Finished chain.
Chatbot: Would you like to ask anything else? If not, could you please provide your feedback and rating out of 5?


In [ ]:
feedback_log

,timestamp,intent,customer_id,feedback,rating
0,2026-05-30 15:44:34.345500,cancel_order,1,Great experience!,4
